# 31. Transfer learning and frozen feature extractors

**Level:** Intermediate

**Prerequisites:** 28: training and model evaluation.

[Course index](00_START_HERE.ipynb) | [Previous](30_RNNs_Sequences_and_Seq2Seq.ipynb) | [Next](32_NLP_Normalization_Stemming_and_Lemmas.ipynb)

**Topics:** Transfer learning, Pretrained features, Fine-tuning, Frozen layers, Domain shift.

**Matching script:** `examples\31_Transfer_Learning_and_Frozen_Features.py`

**How to study:** read the notes, predict an example's output, run it, trace each change in data/shape/state, then complete the practice task. Every worked code cell contains its own imports and data and can run independently of other notebooks.

**Execution contract:** Python 3.11+, course dependencies, CPU only, synthetic/local data. No hosted API requests, model-weight downloads, dataset downloads, listening web servers, credentials, or real device changes. The temporary-file lesson cleans up its own files. Expected errors are caught narrowly and asserted; unexpected errors must remain visible.

## Concept notes and mental model

Transfer learning reuses knowledge learned on a source task for a target task. One approach
freezes an encoder and trains a new head; another fine-tunes some or all layers. Reuse helps
when source features fit target needs, but negative transfer is possible under a mismatch.

Freezing requires_grad prevents parameter gradient updates; eval controls behavior such as
dropout and running statistics. They solve different problems. Make sure the optimizer includes
only the intended parameters and check that frozen weights actually remain unchanged.

This lesson pretrains a tiny source network locally on synthetic data, then transfers its
encoder. It does not download a pretrained model or imply that random features are pretrained.
Large-scale transfer additionally requires model-card and license review, preprocessing
compatibility, target validation, and resource planning.

## Mathematics or data contract

A frozen encoder h=f_theta(x) supplies features; train only a new head g_phi(h) for the target.

## Important limitation

The related synthetic source and target tasks make transfer mechanics visible, not broadly useful representations.

Distinguish what is implemented and asserted below from what would need additional data, training, independent evaluation, authorization, or operational controls.

## Worked example 1: Pretrain locally, freeze, and transfer

**Purpose:** Prove that the source encoder was trained and remains unchanged during target training.

### Read the code in this order
1. Train a small encoder/head on a source regression task.
2. Snapshot and freeze encoder weights.
3. Train a new target head on extracted representations.
4. Assert encoder identity of values and reduced target loss.

### Syntax and behavior to notice
- `backward()` computes/accumulates gradients through the recorded computation graph.
- `zero_grad()` clears earlier gradients before the next independent training update.
- `optimizer.step()` updates only the parameters owned by that optimizer.
- `assert` checks the stated behavior; a failed check raises AssertionError.
- A fixed random seed makes the chosen sampling reproducible within the environment.

### Expected output
This is the output captured from executing the worked example in the recorded environment. Package versions and floating-point rounding can change numeric details; the assertions express the required shapes, invariants, and thresholds.

```text
source loss: 0.007 target loss: 0.037
frozen encoder unchanged
```

**Before running:** identify the input variables, which values are learned versus chosen, and what each assertion proves. A successful shape test does not by itself prove predictive quality. Training-only loss checks are labeled as such.

In [1]:
import torch
from torch import nn

torch.set_num_threads(1)
torch.manual_seed(42)
X = torch.randn(120, 2)
source_y = (X[:, :1] + 0.5 * X[:, 1:])
encoder = nn.Sequential(nn.Linear(2, 6), nn.Tanh())
source_head = nn.Linear(6, 1)
optimizer = torch.optim.Adam(list(encoder.parameters()) + list(source_head.parameters()), lr=0.03)
for _ in range(100):
    optimizer.zero_grad()
    loss = ((source_head(encoder(X)) - source_y) ** 2).mean()
    loss.backward()
    optimizer.step()
assert loss.item() < 0.1
before = [parameter.detach().clone() for parameter in encoder.parameters()]
encoder.requires_grad_(False)
encoder.eval()
features = encoder(X).detach()
target_y = 2 * source_y + 1
target_head = nn.Linear(6, 1)
target_optimizer = torch.optim.Adam(target_head.parameters(), lr=0.05)
initial = ((target_head(features) - target_y) ** 2).mean().item()
for _ in range(100):
    target_optimizer.zero_grad()
    target_loss = ((target_head(features) - target_y) ** 2).mean()
    target_loss.backward()
    target_optimizer.step()
assert target_loss.item() < initial
for saved, parameter in zip(before, encoder.parameters()):
    torch.testing.assert_close(saved, parameter)
print("source loss:", round(loss.item(), 3), "target loss:", round(target_loss.item(), 3))
print("frozen encoder unchanged")

source loss: 0.007 target loss: 0.037
frozen encoder unchanged


### Interpret example 1

The result should demonstrate this purpose: **Prove that the source encoder was trained and remains unchanged during target training.**

Follow the assertions back to the exact data or state being checked. Where a model was fitted, distinguish training fit from held-out evaluation. Where input is rejected, inspect whether rejection happens before any intended mutation or tool execution. Do not delete a failed assertion to make an incorrect result look valid.

## Practice: extend the concept yourself

Compare target performance for a random encoder, frozen pretrained encoder, and fully fine-tuned encoder.

1. Write your expected result before editing code.
2. Add one normal-case test and one boundary/failure-case test.
3. Keep training/validation/test responsibilities separate where relevant.
4. Report an honest failure or unknown result rather than a success-shaped fallback.

The task is intentionally not filled in for you; the worked examples above are complete and runnable. Add your own code cell for the extension.

## Understanding checkpoint
- Explain `Transfer learning` in your own words and give one example of when it is useful.
- Trace one worked example from input through transformation to returned or printed output.
- State this lesson's mathematical/data contract: A frozen encoder h=f_theta(x) supplies features; train only a new head g_phi(h) for the target.
- Name the boundary of the demonstration: The related synthetic source and target tasks make transfer mechanics visible, not broadly useful representations.
- Describe one test that could catch a plausible but wrong implementation.

## Primary references and further reading

- `https://docs.pytorch.org/tutorials/beginner/basics/intro.html`

Research snapshot: **2026-10-17**. Foundational references need not be recent. Mutable library docs describe their current version; compare with the tested dependency versions in the course manifest. References are reading material, not runtime downloads.